# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)


# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType, FloatType
from pyspark.sql.functions import sum as spark_sum, col, cast, weekofyear, max as spark_max

from pyspark.sql.functions import concat, lit

import requests, json, pandas as pd

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
# Write your solution here

#defining the destination location for the parquet file
parquet_location = "/Volumes/training/raw/files/"

# Extract
bookings = spark.read.table("bookings")

# Transform
result = bookings.where(col("starttime").like("2012-09%")).groupBy("facid").agg(spark_sum("slots").alias("Total Slots")).orderBy("Total Slots")
result.display()

# Load
result.write.mode("overwrite").parquet(parquet_location + "bookings.parquet")

facid,Total Slots
5,122
3,422
7,426
8,471
6,540
2,570
1,588
0,591
4,648


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are partitions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS training.transformed;
CREATE VOLUME IF NOT EXISTS training.transformed.files;

In [0]:
# Write your solution here

# Extract
bookings = spark.read.table("bookings")
facilities = spark.read.table("facilities")
members = spark.read.table("members")

# Transform
three_join = bookings.join(members, bookings.memid == members.memid, "left").join(facilities, bookings.facid == facilities.facid, "left")
result = three_join.select(concat(col("members.firstname"), lit(" "), col("members.surname")).alias("member"), col("facilities.name")).distinct().where(col("facilities.name").like("Tennis%")).orderBy("member", "facilities.name")
result.display()

# Load
partitioned_result = result.repartition("name")
partitioned_result.write.saveAsTable("threejoin_delta", mode="overwrite")

# Comparision -- doesn't work in the new serverless setup
# print(f"Original partitioning: {result.rdd.getNumPartitions()}")
# print(f"New partitioning: {partitioned_result.rdd.getNumPartitions()}")


member,name
Anne Baker,Tennis Court 1
Anne Baker,Tennis Court 2
Burton Tracy,Tennis Court 1
Burton Tracy,Tennis Court 2
Charles Owen,Tennis Court 1
Charles Owen,Tennis Court 2
Darren Smith,Tennis Court 2
David Farrell,Tennis Court 1
David Farrell,Tennis Court 2
David Jones,Tennis Court 1


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
# Write your solution here

# Extract
# curl --request GET \
	# --url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=MSFT&outputsize=compact&datatype=json' \
	# --header 'x-rapidapi-host: alpha-vantage.p.rapidapi.com' \
	# --header 'x-rapidapi-key: ***'


url = "https://alpha-vantage.p.rapidapi.com/query"

# create an iterable for companies: Google, Apple, Microsoft, and Tesla
companies = ["GOOG", "AAPL", "MSFT", "TSLA"]
ts_data = []

for company in companies:
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": f"{company}",
        "outputsize": "compact",
        "datatype": "json"
    }

    headers = {
        "x-rapidapi-key": "f092b76a78msh80bac7f8ae329b7p105ea6jsn5ebbe239877d",
        "x-rapidapi-host": "alpha-vantage.p.rapidapi.com"
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()
    ts = data["Time Series (Daily)"]
    
    for date, values in ts.items():
        row = {
            "company": company,
            "date": date,
            "open": values["1. open"],
            "high": values["2. high"],
            "low": values["3. low"],
            "close": values["4. close"],
            "volume": values["5. volume"]
        }
        ts_data.append(row)

pdf = pd.DataFrame(ts_data)

df  = spark.createDataFrame(pdf)

# making sure the data is in a distributed format
# rdd = spark.sparkContext.parallelize(json_data) -- doesn't work in the new serverless setup

# Transform

# creating a "week" column in the df
df = df.withColumn("week", weekofyear(col("date").cast("date")))

result = df.groupBy("company", "week").agg(spark_max("close").alias("max_close"))
result.display()

# Load
partitioned_result = result.repartition("company")
partitioned_result.write.mode("overwrite").saveAsTable("max_closing_price_weekly")


company,week,max_close
AAPL,17,209.2800
AAPL,18,213.3200
AAPL,19,198.8900
AAPL,20,212.9300
AAPL,21,208.7800
AAPL,22,200.8500
AAPL,23,203.9200
AAPL,24,202.6700
AAPL,25,201.0000
AAPL,26,201.5600


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
%pip install psycopg2-binary
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Write your solution here

# Extract
rna_data = spark.read.jdbc(
  url="jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs",
  table="rna",
  properties={"user": "reader", "password": "NWDMCE5xdipIjRrp", "driver": "org.postgresql.Driver"}
)
rna_data = rna_data.limit(100)
rna_data.display()

# Load
rna_data.write.mode("overwrite").saveAsTable("rna_100_records")

id,upi,timestamp,userstamp,crc64,len,seq_short,seq_long,md5
8974816,URS000088F1E0,2015-10-20T18:04:07.000Z,RNACEN,5BA212254E5BC09E,1476,ACGCTGGCGGCGTGCCTAATACATGCAAGTTGAGCGCTGAAGGTTGGTACTTGTACCGACTGGATGAGCAGCGAACGGGTGAGTAACGCGTGGGGAATCTGCCTTTGAGCGGGGGACAACATTTGGAAACGAATGCTAATACCGCATAAAAACTTTAAACACAAGTTTTAAGTTTGAAAGATGCAATTGCATCACTCAAAGATGATCCCGCGTTGTATTAGCTAGTTGGTGAGGTAAAGGCTCACCAAGGCGATGATACATAGCCGACCTGAGAGGGTGATCGGCCACATTGGGACTGAGACACGGCCCAAACTCCTACGGGAGGCAGCAGTAGGGAATCTTCGGCAATGGACGAAAGTCTGACCGAGCAACGCCGCGTGAGTGAAGAAGGTTTTCGGATCGTAAAACTCTGTTGGTAGAGAAGAACGTTGGTGAGAGTGGAAAGCTCATCAAGTGACGGTAACTACCCAGAAAGGGACGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGTCCCGAGCGTTGTCCGGATTTATTGGGCGTAAAGCGAGCGCAGGTGGTTTATTAAGTCTGGTGTAAAAGGCAGTGGCTCAACCATTGTATGCATTGGAAACTGGTAGACTTGAGTGCAGGAGAGGAGAGTGGAATTCCATGTGTAGCGGTGAAATGCGTAGATATATGGAGGAACACCGGTGGCGAAAGCGGCTCTCTGGCCTGTAACTGACACTGAGGCTCGAAAGCGTGGGGAGCAAACAGGATTAGATACCCTGGTAGTCCACGCCGTAAACGATGAGTGCTAGATGTAGGGAGCTATAAGTTCTCTGTATCGCAGCTAACGCAATAAGCACTCCGCCTGGGGAGTACGACCGCAAGGTTGAAACTCAAAGGAATTGACGGGGGCCCGCACAAGCGGTGGAGCATGTGGTTTAATTCGAAGCAACGCGAAGAACCTTACCAGGTCTTGACATACTCGTGCTATTCCTAGAGATAGGAAGTTCCTTCGGGACACGGGATACAGGTGGTGCATGGTTGTCGTCAGCTCGTGTCGTGAGATGTTGGGTTAAGTCCCGCAACGAGCGCAACCCCTATTGTTAGTTGCCATCATTAAGTTGGGCACTCTAACGAGACTGCCGGTGATAAACCGGAGGAAGGTGGGGATGACGTCAAATCATCATGCCCCTTATGACCTGGGCTACACACGTGCTACAATGGATGGTACAACGAGTCGCGAGACAGTGATGTTTAGCTAATCTCTTAAAACCATTCTCAGTTCGGATTGTAGGCTGCAACTCGCCTACATGAAGTCGGAATCGCTAGTAATCGCGGATCAGCACGCCGCGGTGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCACACCACGGGAGTTGGGAGTACCCGAAGTAGGTTGCCTAACCGCAAGGAGGGCGCTTCCTAAGGTAAGACCGATGACTGGGGTGAAGTCGTAACAA,null,974f0301409d15bf3216dd165b706694
8974930,URS000088F252,2015-10-20T18:04:07.000Z,RNACEN,7E8E82EEB670D597,1407,TTAACACATGCAAGTCGAGCGGGCATAGCAATATGTCAGCGGCAGACGGGTGAGTAACGCGTGGGAACGTACCTTTTGGTTCGGAACAACACAGGGAAACTTGTGCTAATACCGGATAAGCCCTTACGGGGAAAGATTTATCGCCGAAAGATCGGCCCGCGTCTGATTAGCTAGTTGGTGAGGTAATGGCTCACCAAGGCGACGATCAGTAGCTGGTCTGAGAGGATGATCAGCCACATTGGGACTGAGACACGGCCCAAACTCCTACGGGAGGCAGCAGTGGGGAATATTGGACAATGGGCGCAAGCCTGATCCAGCCATGCCGCGTGAGTGATGAAGGCCCTAGGGTTGTAAAGCTCTTTTGTGCGGGAAGATAATGACGGTACCGCAAGAATAAGCCCCGGCTAACTTCGTGCCAGCAGCCGCGGTAATACGAAGGGGGCTGGCGTTGCTCGGAATCACTGGGCGTAAAGGGTGCGTAGGCGGGTCTTTAAGTCAGGGGTGAAATCCTGGAGCTCAACTCCAGAACTGCCTTTGATACTGAAGATCTTGAGTTCGGGAGAGGTGAGTGGAACTGCGAGTGTAGAGGTGAAATTCGTAGATATTCGCAAGAACACAAGTGGGCGAAGGCGGCTCACTGGCCCGATACTGACGCTGAGCACGAAAGCGTGGGGAGCAAACAGNATTAGATACCCTGGTAGTCCACGCCGTAAACGATGAATGCCAGCCGNTTAGTGGGTTTACTCACTAGTGACGCAGCTAACGCTTTAAGCATTCCGCCTGGGGAGTACGGTCGCAAGATTAAAACTCAAAGGAATTGACGGGGGCCCGCACAAGCGGTGGAGCATGTGGTTAATTCGACGCAACGCGCAGAACCTTACCAGCCCTTGACATCCCGGTCGCGGACTCCAGAGACGGAGTTCTTCAGTTCGGCTGGACCGGAGACAGGTGCTGCATGGCTGTCGTCAGCTCGTGTCGTGAGATGTTGGGTTAAGTCCCGCAACGAGCGCAACCCCCGTCCTTAGTTGCTACCATTTAGTTGAGCACTCTAAGGAGACTGCCGGTGATAAGCCGCGAGGAAGGTGGGGATGACGTCAAGTCCTCATGGCCCTTACGGGCTGGGCTACACACGTGCTGCAATGGCGGTGACAATGGGATGCTAAGGGGCGACCCTTCGCAAATCTCAAAAAGCCGTCTCAGTTCGGATTGGGCTCTGCAACTCGAGCCCATGAAGTTGGAATCGCTAGTAATCGTGGATCAGCACGCCACGGTGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCACACCATGGGAGTTGGTTTTACCTGAAGACGGTGCGCTAACCGAAAGGGGGCAGCCGGCCACGGTAGGGTCAGCGACTGGGGTGAAGTCGTAACAAGGTA,null,53b857aac944a3c84730f8184471d27f
8974933,URS000088F255,2015-10-20T18:04:07.000Z,RNACEN,98B985F04883C375,1550,TGGAGAGTTTGATCCTGGCTCAGGATGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAGCGAACGGATGAGAAGCTTGCTTCTCTGATGTTAGCGGCGGACGGGTGAGTAACACGTGGATAACCTACCTATAAGACTGGGATAACTTCGGGAAACCGGAGCTAATACCGGATAATATTTTGAACCGCATGGTTCAAAAGTGAAAGACGGTCTTGCTGTCACTTATAGATGGATCCGCGCTGCATTAGCTAGTTGGTAAGGTAACGGCTTACCAAGGCAACGATGCATAGCCGACCTGAGAGGGTGATCGGCCACACTGGAACTGAGACACGGTCCAGACTCCTACGGGAGGCAGCAGTAGGGAATCTTCCGCAATGGGCGAAAGCCTGACGGAGCAACGCCGCGTGAGTGATGAAGGTCTTCGGATCGTAAAACTCTGTTATTAGGGAAGAACATATGTGTAAGTAACTGTGCACATCTTGACGGTACCTAATCAGAAAGCCACGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGTGGCAAGCGTTATCCGGAATTATTGGGCGTAAAGCGCGCGTAGGCGGTTTTTTAAGTCTGATGTGAAAGCCCACGGCTCAACCGTGGAGGGTCATTGGAAACTGGAAAACTTGAGTGCAGAAGAGGAAAGTGGAATTCCATGTGTAGCGGTGAAATGCGCAGAGATATGGAGGAACACCAGTGGCGAAGGCGACTTTCTGGTCTGTA